In [1]:
import pandas as pd
import requests
import time

# ==========================================
# LOAD ALL DISTRICT COORDINATES
# ==========================================

districts = pd.read_csv(
    "../datasets/yield/up_all_district_coordinates.csv"
)

print(districts.head())

# ==========================================
# STORE RESULTS
# ==========================================

all_weather_data = []

# ==========================================
# LOOP THROUGH DISTRICTS
# ==========================================

for index, row in districts.iterrows():

    district = row['District_Name']

    lat = row['Latitude']

    lon = row['Longitude']

    print(f"\nFetching weather data for: {district}")

    # NASA POWER API URL
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        f"?parameters=T2M,RH2M,PRECTOTCORR"
        f"&community=AG"
        f"&longitude={lon}"
        f"&latitude={lat}"
        f"&start=20150101"
        f"&end=20241231"
        f"&format=JSON"
    )

    try:

        response = requests.get(url)

        data = response.json()

        parameters = data['properties']['parameter']

        # Extract yearly averages
        yearly_data = {}

        for date in parameters['T2M']:

            year = int(date[:4])

            if year not in yearly_data:

                yearly_data[year] = {
                    'T2M': [],
                    'RH2M': [],
                    'PRECTOTCORR': []
                }

            yearly_data[year]['T2M'].append(
                parameters['T2M'][date]
            )

            yearly_data[year]['RH2M'].append(
                parameters['RH2M'][date]
            )

            yearly_data[year]['PRECTOTCORR'].append(
                parameters['PRECTOTCORR'][date]
            )

        # Calculate yearly averages
        for year in yearly_data:

            avg_t2m = (
                sum(yearly_data[year]['T2M'])
                / len(yearly_data[year]['T2M'])
            )

            avg_rh2m = (
                sum(yearly_data[year]['RH2M'])
                / len(yearly_data[year]['RH2M'])
            )

            total_rain = (
                sum(yearly_data[year]['PRECTOTCORR'])
            )

            all_weather_data.append({
                'Year': year,
                'T2M': avg_t2m,
                'RH2M': avg_rh2m,
                'PRECTOTCORR': total_rain,
                'District_Name': district
            })

        # Avoid API overload
        time.sleep(1)

    except Exception as e:

        print(f"Error for {district}: {e}")

# ==========================================
# CREATE DATAFRAME
# ==========================================

weather_df = pd.DataFrame(all_weather_data)

print("\nFINAL WEATHER DATASET")

print(weather_df.head())

print("\nTOTAL ROWS:")

print(len(weather_df))

# ==========================================
# SAVE CSV
# ==========================================

weather_df.to_csv(
    "../datasets/weather/full_up_weather.csv",
    index=False
)

print("\nWeather dataset saved successfully!")

    District_Name  Latitude  Longitude
0            Agra   27.1767    78.0081
1         Aligarh   27.8974    78.0880
2       Prayagraj   25.4358    81.8463
3  Ambedkar Nagar   26.4050    82.8390
4          Amethi   26.1542    81.8144

Fetching weather data for: Agra

Fetching weather data for: Aligarh

Fetching weather data for: Prayagraj

Fetching weather data for: Ambedkar Nagar

Fetching weather data for: Amethi

Fetching weather data for: Amroha

Fetching weather data for: Auraiya

Fetching weather data for: Azamgarh

Fetching weather data for: Baghpat

Fetching weather data for: Bahraich

Fetching weather data for: Ballia

Fetching weather data for: Balrampur

Fetching weather data for: Banda

Fetching weather data for: Barabanki

Fetching weather data for: Bareilly

Fetching weather data for: Basti

Fetching weather data for: Bhadohi

Fetching weather data for: Bijnor

Fetching weather data for: Budaun

Fetching weather data for: Bulandshahr

Fetching weather data for: Chandauli


In [ ]:

+